In [ ]:
import numpy as np
import pickle
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PyEMD import EMD
from scipy.signal import welch
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. 特征提取模块
# ==========================================
def get_psd_features(signal, fs=128):
    freqs, psd = welch(signal, fs, nperseg=fs*2 if len(signal)>fs*2 else len(signal))
    bands = {'Theta': (4, 8), 'Alpha': (8, 13), 'Beta': (13, 30), 'Gamma': (30, 45)}
    return [np.mean(psd[np.logical_and(freqs >= b[0], freqs <= b[1])]) if np.any(np.logical_and(freqs >= b[0], freqs <= b[1])) else 0 for b in bands.values()]

def extract_features(segment, emd_tool):
    all_ch_feats = []
    for ch_data in segment:
        imfs = emd_tool.emd(ch_data, max_imf=1)
        d_t = np.mean(np.abs(np.diff(imfs[0]))) if imfs.shape[0]>0 else 0
        psd_f = get_psd_features(ch_data)
        all_ch_feats.extend([d_t] + psd_f)
    return all_ch_feats

# ==========================================
# 2. 实验引擎
# ==========================================
def run_experiment(data_dir):
    subject_files = sorted([f for f in os.listdir(data_dir) if f.endswith('.dat')])
    if not subject_files: raise FileNotFoundError("未找到.dat文件，请检查路径。")
    
    emd = EMD()
    dims = {0: "Valence", 1: "Arousal", 2: "Dominance", 3: "Liking"}
    clfs_pool = {
        "Linear_SVM": svm.SVC(kernel='linear'),
        "RBF_SVM":    svm.SVC(kernel='rbf'),
        "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42)
    }
    
    raw_results = []
    ch_idx = [0, 16, 3, 20, 7, 25, 11, 29]
    
    print(f">>> 开始 32 名受试者定量分析...")
    for f_name in subject_files:
        print(f"正在处理: {f_name}       ", end='\r')
        with open(os.path.join(data_dir, f_name), 'rb') as f:
            data_dict = pickle.load(f, encoding='latin1')
        
        X_sub = []
        for i in range(40):
            vid_sig = data_dict['data'][i, ch_idx, :]
            for s in range(0, 8064 - 672 + 1, 672):
                X_sub.append(extract_features(vid_sig[:, s:s+672], emd))
        X_sub = StandardScaler().fit_transform(np.array(X_sub))
        
        for d_idx, d_name in dims.items():
            y_sub = np.array([1 if data_dict['labels'][i, d_idx] >= 5 else 0 for i in range(40) for _ in range(12)])
            if len(np.unique(y_sub)) < 2: continue 
            
            for c_name, clf in clfs_pool.items():
                y_pred = cross_val_predict(clf, X_sub, y_sub, cv=5)
                raw_results.append({
                    'Subject': f_name, 'Dimension': d_name, 
                    'Classifier': c_name, 'Accuracy': accuracy_score(y_sub, y_pred),
                    'CM': confusion_matrix(y_sub, y_pred)
                })
    return pd.DataFrame(raw_results)

# ==========================================
# 3. 绘图与保存模块 (修正 Average Acc 显示)
# ==========================================
def save_all_visualizations(df, output_folder="Experiment_Results"):
    if not os.path.exists(output_folder): os.makedirs(output_folder)
    
    print("\n>>> 正在批量生成 12 组混淆矩阵对比图...")
    for dim in df['Dimension'].unique():
        for clf in df['Classifier'].unique():
            sub_df = df[(df['Dimension'] == dim) & (df['Classifier'] == clf)].sort_values('Accuracy')
            if sub_df.empty: continue
            
            # 筛选 Best, Worst, Median
            best = sub_df.iloc[-1]
            worst = sub_df.iloc[0]
            median = sub_df.iloc[len(sub_df)//2]
            
            # 计算全组平均矩阵和平均准确率
            avg_cm = np.mean(np.stack(sub_df['CM'].values), axis=0)
            mean_acc_val = sub_df['Accuracy'].mean()

            fig, axes = plt.subplots(1, 4, figsize=(22, 5))
            cms = [best['CM'], worst['CM'], median['CM'], avg_cm]
            titles = ["Best Case", "Worst Case", "Median Case", "Average (Overall)"]
            
            for i in range(4):
                sns.heatmap(cms[i], annot=True, fmt='.1f' if i==3 else 'd', cmap='YlGnBu', ax=axes[i])
                
                # 逻辑修正：前三张图取个体值，第四张图取平均值
                if i < 3:
                    # 索引映射：0->Worst, 1->Best, 2->Median (根据前面排序逻辑)
                    idx_map = [0, -1, len(sub_df)//2]
                    current_acc = sub_df.iloc[idx_map[i if i==0 else (1 if i==1 else 2)]]['Accuracy']
                    axes[i].set_title(f"{titles[i]}\nAcc: {current_acc:.2f}")
                else:
                    axes[i].set_title(f"{titles[i]}\nMean Acc: {mean_acc_val:.2f}")
                
                axes[i].set_xlabel('Predicted')
                axes[i].set_ylabel('Actual')
            
            plt.suptitle(f"Confusion Matrices Analysis: {dim} using {clf}", fontsize=16, fontweight='bold')
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.savefig(f"{output_folder}/CM_{dim}_{clf}.png")
            plt.close()

    # 方案 B: 全维度汇总图
    print(">>> 正在生成汇总对比柱状图...")
    plt.figure(figsize=(12, 7))
    summary = df.groupby(['Dimension', 'Classifier'])['Accuracy'].mean().unstack()
    ax = summary.plot(kind='bar', edgecolor='black', rot=0, figsize=(12, 7), colormap='Set3')
    plt.title("Mean Accuracy Across All Dimensions & Classifiers", fontsize=14)
    plt.ylabel("Accuracy Score")
    plt.ylim(0, 1.0)
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.legend(title="Classifier", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(f"{output_folder}/Overall_Comparison_Bar.png")
    plt.show()

# ==========================================
# 4. 主程序
# ==========================================
if __name__ == "__main__":
    INPUT_DIR = "./data_preprocessed_python/" # 确保此路径下有 s01.dat 等文件
    
    try:
        final_df = run_experiment(INPUT_DIR)
        
        if not final_df.empty:
            final_df.to_csv("DEAP_Full_Results.csv", index=False)
            print("\n[数据成功] 原始明细已保存至 DEAP_Full_Results.csv")
            
            save_all_visualizations(final_df, output_folder="Experiment_Results")
            print(f"[图表成功] 所有分析图已保存至 'Experiment_Results' 文件夹")
        else:
            print("\n[警告] 未能生成有效结果，请检查数据路径或标签格式。")
            
    except Exception as e:
        print(f"\n[运行崩溃] 错误信息: {e}")